## FINAL CODE

## YOLO v12 + SAM

In [2]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os
from accuracy_indian import compute_metrics
from generate_isolated_masks import generate_isolated_mask

In [4]:
# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/yolov12/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/SAM_MODEL/Final_Models/FineTune_model_epoch_30_27_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_YOLO_v12_SAM_PreTrain.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

images_dirs = sorted(os.listdir(data_folder))
masks_dirs = sorted(os.listdir(mask_folder))

for i in range(len(os.listdir(data_folder))):
    img_name = images_dirs[i]
    mask_name = masks_dirs[i]

    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    
    mask_path = os.path.join(mask_folder, mask_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)


image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_107_jpg.rf.bc49affc34308ae5803f34db217fa6d4.jpg: 1024x1024 86 Rooftops, 3.4ms
Speed: 1.3ms preprocess, 3.4ms inference, 0.4ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_116_jpg.rf.9f22d446fa753ea32027f482d1531346.jpg: 1024x1024 79 Rooftops, 3.8ms
Speed: 1.7ms preprocess, 3.8ms inference, 0.4ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_127_jpg.rf.8618846b9a9e0b0204f34171598a72f5.jpg: 1024x1024 80 Rooftops, 3.6ms
Speed: 1.9ms preprocess, 3.6ms inference, 0.4ms postprocess per image

In [ ]:
len(os.listdir(data_folder))

In [5]:
metrics_df.head(5)

,pixel_iou,pixel_dice,pixel_accuracy,pixel_precision,pixel_recall,region_iou,region_dice,region_precision,region_recall,region_success_accuracy
0,0.708165,0.829153,0.998694,0.779995,0.884923,0.708165,0.829153,0.779995,0.884923,1.0
1,0.841278,0.913798,0.999403,0.899187,0.928891,0.841278,0.913798,0.899187,0.928891,1.0
2,0.703023,0.825618,0.999091,0.721920,0.964103,0.703023,0.825618,0.721920,0.964103,1.0
3,0.734930,0.847215,0.998511,0.753482,0.967583,0.734930,0.847215,0.753482,0.967583,1.0
4,0.812248,0.896399,0.999155,0.849701,0.948528,0.812248,0.896399,0.849701,0.948528,1.0


### This is for YOLO v12 + SAM

In [6]:
metrics_df.mean()

pixel_iou                  0.696856
pixel_dice                 0.775506
pixel_accuracy             0.999034
pixel_precision            0.739306
pixel_recall               0.923252
region_iou                 0.696856
region_dice                0.775506
region_precision           0.739306
region_recall              0.923252
region_success_accuracy    0.868654
dtype: float64

### lets test for single image as whole.

In [ ]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from accuracy import compute_metrics
from generate_isolated_masks import generate_isolated_mask
import matplotlib.pyplot as plt 



# Load YOLO model 
yolo_model = YOLO("/home/selc-a4-sr2/yolo_v12/yolov12/runs/detect/train4/weights/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/selc-a4-sr2/Solar_Rooftop_Detection/checkpoints/Final_Models/FineTune_model_3_epoch_20_03_03_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)


test_image = cv2.imread("/home/selc-a4-sr2/Solar_Rooftop_Detection/Arial_validation_images/images/1.jpg")
yolo_results = yolo_model.predict(test_image, conf=0.5, device="cuda")
boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
# print(len(boxes))
sam_predictor.set_image(test_image)
masks, _, _ = sam_predictor.predict(box=boxes[3])
masks
plt.imshow(masks[0], cmap="gray")


In [ ]:
cv2.rectangle(test_image, (int(boxes[3][0]), int(boxes[3][1])), (int(boxes[3][2]), int(boxes[3][3])), (0, 255, 0), 3)
plt.imshow(test_image[:,:,::-1])

In [ ]:
original_mask = cv2.imread("/home/selc-a4-sr2/Solar_Rooftop_Detection/Arial_validation_images/masks/1.jpg", cv2.IMREAD_GRAYSCALE)
plt.imshow(original_mask, cmap="gray")

# try to test on indian image

In [ ]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from accuracy import compute_metrics
from generate_isolated_masks import generate_isolated_mask
import matplotlib.pyplot as plt 



# Load YOLO model 
yolo_model = YOLO("/home/selc-a4-sr2/yolo_v12/yolov12/runs/detect/train4/weights/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/selc-a4-sr2/Solar_Rooftop_Detection/checkpoints/Final_Models/FineTune_model_3_epoch_20_03_03_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)


test_image = cv2.imread("/home/selc-a4-sr2/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Demo_images/sample_1.jpg")
yolo_results = yolo_model.predict(test_image, conf=0.5, device="cuda")
boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
print(len(boxes))
sam_predictor.set_image(test_image)

demo = test_image.copy()
for i in boxes:
    cv2.rectangle(demo, (int(i[0]), int(i[1])), (int(i[2]), int(i[3])), (0, 255, 0), 3)

plt.imshow(demo[:,:,::-1])

In [ ]:
for i in boxes:
    masks, _, _ = sam_predictor.predict(box=i)
    plt.imshow(masks[0], cmap="gray")
    plt.show()

# YOLO v11 + SAM

In [8]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from accuracy_indian import compute_metrics
from generate_isolated_masks import generate_isolated_mask

In [12]:
# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/indian_weightss/yolov11_indian.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/SAM_MODEL/Final_Models/FineTune_model_epoch_30_27_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)


results_data = []

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_YOLO_v11_SAM_PreTrain.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

images_dirs = sorted(os.listdir(data_folder))
masks_dirs = sorted(os.listdir(mask_folder))

for i in range(len(os.listdir(data_folder))):
    img_name = images_dirs[i]
    mask_name = masks_dirs[i]
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    mask_path = os.path.join(mask_folder, mask_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)


image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_107_jpg.rf.bc49affc34308ae5803f34db217fa6d4.jpg: 1024x1024 81 Rooftops, 2.9ms
Speed: 1.3ms preprocess, 2.9ms inference, 0.4ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_116_jpg.rf.9f22d446fa753ea32027f482d1531346.jpg: 1024x1024 77 Rooftops, 2.9ms
Speed: 1.3ms preprocess, 2.9ms inference, 0.4ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_127_jpg.rf.8618846b9a9e0b0204f34171598a72f5.jpg: 1024x1024 82 Rooftops, 3.1ms
Speed: 1.2ms preprocess, 3.1ms inference, 0.5ms postprocess per image

In [13]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_YOLO_v11_SAM_PreTrain.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.697668
pixel_dice                 0.775925
pixel_accuracy             0.999046
pixel_precision            0.740434
pixel_recall               0.924862
region_iou                 0.697668
region_dice                0.775925
region_precision           0.740434
region_recall              0.924862
region_success_accuracy    0.868720
dtype: float64

# YOLO v10

In [14]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from accuracy_indian import compute_metrics
from generate_isolated_masks import generate_isolated_mask

In [15]:
# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/indian_weightss/yolov10_indian.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/SAM_MODEL/Final_Models/FineTune_model_epoch_30_27_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

results_data = []

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_YOLO_v10_SAM_PreTrain.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

images_dirs = sorted(os.listdir(data_folder))
masks_dirs = sorted(os.listdir(mask_folder))

for i in range(len(os.listdir(data_folder))):
    img_name = images_dirs[i]
    mask_name = masks_dirs[i]
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    mask_path = os.path.join(mask_folder, mask_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)


image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_107_jpg.rf.bc49affc34308ae5803f34db217fa6d4.jpg: 1024x1024 79 Rooftops, 2.8ms
Speed: 1.3ms preprocess, 2.8ms inference, 0.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_116_jpg.rf.9f22d446fa753ea32027f482d1531346.jpg: 1024x1024 75 Rooftops, 2.9ms
Speed: 1.2ms preprocess, 2.9ms inference, 0.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test/images/sample_127_jpg.rf.8618846b9a9e0b0204f34171598a72f5.jpg: 1024x1024 73 Rooftops, 2.8ms
Speed: 1.4ms preprocess, 2.8ms inference, 0.2ms postprocess per image

In [16]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/Results/validation_results_YOLO_v10_SAM_PreTrain.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.716876
pixel_dice                 0.798337
pixel_accuracy             0.999031
pixel_precision            0.759461
pixel_recall               0.926752
region_iou                 0.716876
region_dice                0.798337
region_precision           0.759461
region_recall              0.926752
region_success_accuracy    0.895957
dtype: float64